# পাঠ ১৮ (পরবর্তী): রসিদ যা প্রমাণ করে একটি *মানব* ক্রিয়াটি অনুমোদন করেছে

এই পাঠটি প্রমাণ করে **নির্বাহী** কী করেছিল এবং **গেট** কী সিদ্ধান্ত নিয়েছিল। এই নোটবুকটি অনুপস্থিত অর্ধেক যোগ করছে: প্রমাণ যে একটি **নির্দিষ্ট নামকৃত মানব** **সঠিক** ক্রিয়াটি অনুমোদন করেছে — সম্পূর্ণ প্রচলিত ক্রিয়ার উপর একটি পৃথক, মানব-ধারণকৃত স্বাক্ষর, অফলাইন যাচাই করা হয়েছে।

উভয় আর্টিফ্যাক্ট এখানে ব্যবহার করে **পাঠের রসিদগুলির মতো একই খাম আকৃতি**: একটি সমতল পেলোড যার একটি `type` ক্ষেত্র রয়েছে, যা ইড২৫৫১৯ দ্বারা সরাসরি প্রচলিত JCS বাইটগুলোর উপর স্বাক্ষরিত, এবং একটি কাঠামোবদ্ধ `signature` অবজেক্ট সংযুক্ত (এবং স্বাক্ষরিত বাইট থেকে বাদ দেওয়া হয়েছে)। অনুমোদন রসিদ একটি নতুন `type` (`human.approval.v1`) যেটি ক্রিয়ার ধরনের পাশাপাশি, তাই একটি `verify_chain` একই কোডপথ দিয়ে উভয় আর্টিফ্যাক্ট ধরনকেই কভার করে যা আপনি প্রধান নোটবুকে তৈরি করেছিলেন। এই মানব-অনুমোদন রসিদটি এখানে সংজ্ঞায়িত একটি শিক্ষামূলক রচনা, নয় যে ড্রাফট-ফারলি-আক্টা-স্বাক্ষরিত-রসিদ দ্বারা সংজ্ঞায়িত রসিদ ধরণ।

প্রধান নোটবুকের ডেমো যাচাইকারী থেকে একটি ইচ্ছাকৃত আপগ্রেড: যাচাইকারী এখানে `signature.key_id` একটি **স্থিরকৃত মূল নিবন্ধনের বিরুদ্ধে** নিবারণ করে, রসিদের ভেতরে বহন করা পাবলিক কী-র পরিবর্তে বিশ্বাস করে না। এটা হল প্রোডাকশন অবস্থান যা পাঠের নিজস্ব চেকলিস্ট সুপারিশ করে ("যাচাইয়ের পাবলিক কী প্রকাশ করুন"), এবং এটিই جعل করা অস্বীকার করা হয় যাতে পরিবর্তে নিজের কী নিয়ে আসার সুযোগ বন্ধ হয়।

এই নোটবুক যে নিয়ম শেখায়: **একটি স্বাক্ষরিত অনুমোদন নিজেই কর্তৃত্ব নয়।** কর্তৃত্ব তখনই থাকে যখন অনুমোদন রসিদ এবং ক্রিয়া রসিদ একই প্রচলিত ক্রিয়ার সাথে বর্তমান নীতির সংস্করণ, কী, ও মেয়াদোত্তীর্ণ ছাড়াও বন্ধ থাকে, এবং অনুমোদন ইতিমধ্যে ব্যবহৃত না হয়। প্রতিটি ব্যর্থতা একটি **ভিন্ন কারণ** দিয়ে প্রত্যাখ্যাত হয়, তাই আপনি বলতে পারবেন *কর্তৃত্ব স্থায়ী ছিল না* এবং *নির্বাহিত ক্রিয়াটি পরিবর্তিত হয়েছে* এদের মধ্যে পার্থক্য। 


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## সঠিক ক্রিয়া

অনুমোদনের ইউনিট হল **ক্যানোনিক্যাল ক্রিয়া অবজেক্ট** — "অ্যাপ্রুভ রিফান্ড" এর মতো অস্পষ্ট লেবেল নয়, বরং সুনির্দিষ্ট, সম্পূর্ণ নির্দিষ্টকৃত ক্রিয়া। পুরো অবজেক্টে সাইন করা (এবং তা থেকে একটি ডাইজেস্ট তৈরি করা) আমাদের পরে প্রমাণ করতে দেয় যে মানুষটি *এই* অনুমোদন দিয়েছে, অন্য কিছু নয়।


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## এক এনভেলপ, দুই কর্তৃপক্ষ

প্রতিটি রসিদ হলো পাঠের এনভেলপ: একটি সমতল পে-লোড যার মধ্যে একটি `type` ক্ষেত্র রয়েছে, সঙ্গে একটি `signature` অবজেক্ট (`alg`, `sig`, `key_id`) যা স্বাক্ষরিত বাইটের অংশ নয়। `verify_envelope` হলো উভয় রসিদের জন্য একটি ভাগ করা গঠনমূলক + স্বাক্ষর পরীক্ষা; এটি কোন **পিন করা কীগুলি নিবন্ধন** এর বিরুদ্ধে `signature.key_id` সমাধান করে তা কর্তৃপক্ষদের পৃথক রাখে:

- **অনুমোদন রসিদ** (`human.approval.v1`) — নামকৃত অনুমোদনকারী, সম্পূর্ণ মানলিপিক অ্যাকশন **এবং তার ডাইজেস্ট**, `policy_version`, ইস্যু + মেয়াদ শেষ হওয়ার টাইমস্ট্যাম্প। একবারের ব্যবহারের ট্র্যাকিং চেইন স্তরেই করা হয়।
- **অ্যাকশন রসিদ** (`agent.action.v1`) — এজেন্ট পরিচিতি, `run_id`, একই মানলিপিক অ্যাকশন **ডাইজেস্ট**, কর্ম সম্পাদনের ফলাফল + টাইমস্ট্যাম্প, এবং `parent_approval_ref`: অনুমোদনের `receipt_hash`, পাঠের চেইনের `previous_receipt_hash` এর মত একই রীতিনীতি।

ভাগ করা `action_digest` ক্ষেত্র হলো যেই মিলনের উপর সংযোগ নির্ভর করে। `key_id` স্বাক্ষর অবজেক্টে শুধুমাত্র একটি অনুসন্ধান সংকেত হিসেবে থাকে: এটিকে ভিন্ন পিন করা কীর দিকে নির্দেশ করলে স্বাক্ষর পরীক্ষা ফেল করে, তাই এটি কিছু বহন করে না।


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: যেখানে বাস্তবে বেঁধে দেওয়া হয়  

`verify_chain` দুটি সাইনেচার যাচাইয়ের উপরে কোনও সুবিধাজনক মোড়ক **না**। এটি একমাত্র জায়গা যেখানে ভাগ করা প্রামাণিক `action_digest`, নীতি/চাবি/মেয়াদ উত্তীর্ণের **তাজাত্বতা** অনুমোদন, এবং অনুমোদনের **একবার ব্যবহার** একসাথে পরীক্ষা করা হয়, সেই ক্রিয়াকলাপের বিরুদ্ধে যা *এখনই* সম্পাদিত হচ্ছে।  

প্রতিটি ব্যর্থতায় একটি **স্বতন্ত্র কারণ** থাকে, তাই প্রত্যাখ্যানের পাঠক বুঝতে পারে কর্তৃত্ব মেয়াদোত্তীর্ণ হয়েছে কি না (নীতি পরিবর্তিত হয়েছে, চাবি পরিবর্তিত হয়েছে, অনুমোদন মেয়াদোত্তীর্ণ হয়েছে, অনুমোদন ব্যবহৃত হয়েছে) অথবা সম্পাদিত কাজ এখনও বৈধ অনুমোদনের আওতায় পরিবর্তিত হয়েছে (ডাইজেস্ট প্রতিস্থাপন)।  


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## বাইন্ডিং কী ধরেছে

নিচের প্রতিটি ক্ষেত্রে **পতাকৃত** হয় **একটি স্বতন্ত্র কারণে**। প্রথম ব্লকটি ক্লাসিক সেট (তামপার, বিভ্রান্ত প্রতিনিধি, রেপ্লে, কর্তৃপক্ষের যেকোনো পাশে জালিয়াতি, খারাপ ইনপুট) । দ্বিতীয় ব্লকটি হলো সেই জোড়া যা বৈশিষ্ট্যটিকে সত্যিকারের করে তোলে, শুধুমাত্র দিয়েও না:

- **অস্তିত্বরত নয় এমন কর্তৃপক্ষ** — স্বাক্ষর এখনও বৈধ, কিন্তু নীতির সংস্করণ পরিবর্তিত হয়েছে, অনুমোদকের চাবিটি পিন করা রেজিস্ট্রি থেকে সরিয়ে ফেলা হয়েছে, অথবা অনুমোদন কার্যাবলীর পূর্বে মেয়াদ শেষ হয়েছে;
- **ডাইজেস্ট প্রতিস্থাপন** — একটি বৈধভাবে স্বাক্ষরিত অ্যাকশন রসিদ যার `parent_approval_ref` একটি *বাস্তব* অনুমোদনের দিকে নির্দেশ করে, কিন্তু সেই অনুমোদনের প্রচলিত অ্যাকশন ডাইজেস্ট কার্যত চলমান অ্যাকশনের সাথে মেলে না।


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## এটা কি প্রমাণ করে — এবং কি প্রমাণ করে না

**প্রমাণ করে:** একটি নামকৃত মানুষ *এই নির্দ্দিষ্ট মূল কার্যকলাপটি* (সম্পূর্ণ কার্যকলাপ + ডাইজেস্ট, একটি পিন করা রেজিস্ট্রি থেকে নির্ধারিত কী দ্বারা স্বাক্ষরিত) অনুমোদন করেছেন, এবং এজেন্ট *ঠিক সেই অনুমোদিত কার্যকলাপটি* বাস্তবায়ন করেছে (একই ডাইজেস্ট, রসিদ `receipt_hash` দ্বারা অনুমোদনের সাথে যুক্ত, পাঠের নিজের চেইন রীতি অনুযায়ী) — যখন অনুমোদনের নীতির সংস্করণ, কী এবং সময়সীমা তখনো বর্তমান ছিল, ঠিক একবার। যদি কোনও পক্ষ পরিবর্তিত হয়, চেইন বন্ধ হয়ে যায়, এবং প্রত্যাখ্যানের কারণ আপনাকে বলে **কোনটি** বৈশিষ্ট্য ভেঙেছে: পুরানো কর্তৃত্ব বনাম পরিবর্তিত কার্যকলাপ।

**প্রমাণ করে না:** যে অনুমোদন UI মানুষকে দেখিয়েছিল তারা যা স্বাক্ষর করতে যাচ্ছিল তার ঠিক তেমন (WYSIWYS নিজেই একটি সমস্যা), যে কীটি রোটেশনের আগে জোরপূর্বক বা চুরি হয়নি, বা যে আমাদের পরে প্রভাব কার্যকলাপের সাথে মিলে গেছে। স্বাক্ষরিত = অনুমোদিত নয়: একটি অবসন্ন নীতির উপর বৈধ স্বাক্ষর, একটি ঘোরানো কী, একটি মেয়াদোত্তীর্ণ উইন্ডো, বা অন্য ডাইজেস্ট এখানে কিছুই প্রদান করে না।

দুই রসিদ প্রকারই পাঠের এনভেলপ এবং একটি `verify_chain` কোড পাথ ভাগ করে উদ্দেশ্যমূলকভাবে: আপনি মূল নোটবুকে কার্যকলাপ রসিদের জন্য যে বাইন্ডিং তৈরি করেছেন তা হল একই কোড যা মানুষের অনুমোদন পরীক্ষা করে। একটি যাচাইকরণ চুক্তি, পৃথক পিন করা কর্তৃপক্ষ, মূল কার্যকলাপ ডাইজেস্ট দ্বারা মিলিত এবং আর কিছু নয়।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
